# 01. 워크플로우 vs 에이전트: 패턴 선택 가이드

> 같은 LangGraph로도 **워크플로우(고정 경로)** 와 **에이전트(LLM이 동적 결정)** 를 모두 만들 수 있어요. 6가지 대표 워크플로우 패턴을 직접 구현하며 둘의 경계를 그어봐요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **워크플로우(Workflow)**와 **에이전트(Agent)**의 개념적 차이를 설명할 수 있어요
2. 6가지 워크플로우 패턴(Augmented LLM, Prompt Chaining, Parallelization, Routing, Orchestrator-Worker, Evaluator-Optimizer)을 코드로 구현할 수 있어요
3. Agent 패턴의 핵심 구조(tool loop, should_continue)를 이해하고 구현할 수 있어요
4. 주어진 요구사항에 맞는 적절한 패턴을 선택할 수 있어요

## 사전 지식

- Part 02에서 배운 `StateGraph`, `Node`, `Edge`, `START/END` 개념
- `add_messages` reducer와 `MessagesState` 활용법
- `InMemorySaver`를 사용한 체크포인터 기본 사용법


## 워크플로우와 에이전트란?

LLM 애플리케이션을 설계할 때 두 가지 큰 범주로 나뉘어요:

| 구분 | 워크플로우(Workflow) | 에이전트(Agent) |
|------|--------------------|-----------------|
| 제어 흐름 | **사전 정의된 고정 경로** | **LLM이 런타임에 동적으로 결정** |
| 예측 가능성 | 높음 | 낮음 |
| 유연성 | 낮음 | 높음 |
| 적합한 작업 | 명확한 단계, 반복 가능 | 복잡하고 오픈엔드한 작업 |
| 비용/지연 | 통제 가능 | 예측 어려움 |
| 비유 | 🏭 공장 조립 라인 (정해진 순서) | 🧑‍🍳 셰프 (재료 보고 즉흥 요리) |

아래 다이어그램으로 전체 패턴 지형을 살펴볼게요:

```mermaid
flowchart TD
    ROOT["LLM 시스템 아키텍처"] --> WF["워크플로우 Workflow<br/>고정된 제어 흐름"]
    ROOT --> AG["에이전트 Agent<br/>LLM이 흐름 결정"]

    WF --> P1["Augmented LLM<br/>도구+검색+메모리"]
    WF --> P2["Prompt Chaining<br/>순차 체이닝"]
    WF --> P3["Parallelization<br/>병렬 처리"]
    WF --> P4["Routing<br/>입력 분기"]
    WF --> P5["Orchestrator-Worker<br/>동적 태스크 분배"]
    WF --> P6["Evaluator-Optimizer<br/>자기 개선 루프"]

    AG --> A1["Tool Loop<br/>도구 호출 반복"]

    classDef root fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef workflow fill:#cce5ff,stroke:#007bff,color:#004085
    classDef agent fill:#d4edda,stroke:#28a745,color:#155724
    classDef pattern fill:#fff3cd,stroke:#ffc107,color:#856404

    class ROOT root
    class WF workflow
    class AG agent
    class P1,P2,P3,P4,P5,P6,A1 pattern
```


## 환경 설정


In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어와요)
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# LangSmith 추적 설정 (학습 중 그래프 실행 과정을 추적할 수 있어요)
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Part03-Patterns"


## 1. 패턴 1: Augmented LLM (증강 LLM)

가장 기본적인 패턴이에요. LLM에 **도구(Tools)**, **검색(Retrieval)**, **메모리(Memory)** 를 추가해서 능력을 확장해요.

```mermaid
flowchart LR
    IN["입력"] --> LLM["LLM"]
    LLM <--> T["도구 Tools"]
    LLM <--> R["검색 Retrieval"]
    LLM <--> M["메모리 Memory"]
    LLM --> OUT["출력"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class IN input
    class LLM process
    class OUT output
    class T,R,M storage
```


In [ ]:
# LangChain V1 핵심 임포트
from langchain.chat_models import init_chat_model
from langchain.tools import tool

# 기본 모델 초기화: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델 사용 예시:
#   - Anthropic: "anthropic:claude-sonnet-4-5"
#   - Google Gemini: "google_genai:gemini-2.0-flash"
model = init_chat_model("openai:gpt-4o-mini")


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Augmented LLM: 도구 정의 및 모델에 바인딩
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Augmented LLM 테스트: 도구 호출 응답 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 패턴 2: Prompt Chaining (프롬프트 체이닝)

### 왜 필요한가요?

하나의 복잡한 프롬프트로 모든 것을 처리하려고 하면 LLM이 혼란스러워하고, 결과의 품질이 들쑥날쑥해요. 마치 **한 사람에게 동시통역 + 요약 + 교정을 동시에 시키는 것**과 같아요. 각 전문가에게 한 가지씩 맡기는 것이 훨씬 신뢰성이 높죠.

작업을 **여러 순차적 단계**로 분해하여, 각 LLM 호출의 출력이 다음 단계의 입력이 되는 패턴이에요.

```mermaid
flowchart LR
    IN["입력"] --> S1["LLM 1<br/>초안 작성"]
    S1 --> G1{"품질 게이트 1"}
    G1 -->|통과| S2["LLM 2<br/>번역"]
    G1 -->|실패| END1["종료"]
    S2 --> G2{"품질 게이트 2"}
    G2 -->|통과| S3["LLM 3<br/>최종 검토"]
    G2 -->|실패| END2["종료"]
    S3 --> OUT["출력"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef gate fill:#f8d7da,stroke:#dc3545,color:#721c24

    class IN input
    class S1,S2,S3 process
    class OUT output
    class G1,G2 gate
```


In [ ]:
# LangGraph 핵심 구성 요소 임포트
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain.messages import SystemMessage, HumanMessage


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Prompt Chaining: 3단계 콘텐츠 생성 파이프라인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → draft → translate → summarize → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Prompt Chaining 실행 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 패턴 3: Parallelization (병렬화)

독립적인 작업을 **동시에** 실행하여 처리 속도를 높이는 패턴이에요. LangGraph에서는 분기(Fan-out)와 합류(Fan-in) 구조로 구현해요.

```mermaid
flowchart TD
    IN["입력"] --> SPLIT["Fan-out<br>분기"]
    SPLIT --> T1["LLM 1<br>감성 분석"]
    SPLIT --> T2["LLM 2<br>키워드 추출"]
    SPLIT --> T3["LLM 3<br>카테고리 분류"]
    T1 --> MERGE["Fan-in<br>합류"]
    T2 --> MERGE
    T3 --> MERGE
    MERGE --> OUT["통합 결과"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class IN input
    class SPLIT,T1,T2,T3 process
    class MERGE,OUT output
```


In [ ]:
import operator
from typing import Annotated

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Parallelization: 텍스트 다중 분석 파이프라인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → analyze_sentiment + extract_keywords + classify_category → aggregate → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 병렬 분석 실행
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 패턴 4: Routing (라우팅)

입력을 분류하여 **적절한 전문 처리 경로**로 보내는 패턴이에요. 각 경로는 해당 유형에 최적화된 LLM 또는 로직을 사용해요.

```mermaid
flowchart TD
    IN["입력"] --> ROUTER["분류기 LLM<br>Router"]
    ROUTER -->|"기술 문의"| T1["기술 전문 LLM"]
    ROUTER -->|"결제 문의"| T2["결제 전문 LLM"]
    ROUTER -->|"일반 문의"| T3["일반 응답 LLM"]
    T1 --> OUT["최종 응답"]
    T2 --> OUT
    T3 --> OUT

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef router fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class IN input
    class ROUTER router
    class T1,T2,T3 process
    class OUT output
```


In [ ]:
from typing import Literal

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Routing: 고객 문의 유형별 분기 처리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → classify_inquiry → tech_support/billing_support/general_support → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 라우팅 테스트: 다양한 문의 유형 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 패턴 5: Orchestrator-Worker (오케스트레이터-워커)

### 왜 필요한가요?

앞서 배운 Parallelization은 **분석 항목이 미리 정해져 있을 때** 유용해요. 하지만 "이 주제를 깊이 분석해줘"처럼 **몇 개의 서브태스크가 필요한지 미리 알 수 없는 경우**가 있어요. 이때 LLM이 직접 태스크를 분해하고 배분하는 오케스트레이터-워커 패턴이 필요해요. 마치 **프로젝트 매니저**가 업무를 파악하고 팀원에게 배분하는 것과 같아요.

중앙 오케스트레이터 LLM이 태스크를 동적으로 분해하고, 워커들에게 **병렬로 배분**하는 패턴이에요. LangGraph의 `Send` API를 사용해서 동적으로 워커를 생성해요.

```mermaid
flowchart TD
    IN["입력"] --> ORCH["오케스트레이터<br/>Orchestrator LLM"]
    ORCH -->|"Send API"| W1["워커 1<br/>Worker"]
    ORCH -->|"Send API"| W2["워커 2<br/>Worker"]
    ORCH -->|"Send API"| W3["워커 3<br/>Worker"]
    W1 --> SYNTH["합성<br/>Synthesizer"]
    W2 --> SYNTH
    W3 --> SYNTH
    SYNTH --> OUT["최종 결과"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef orch fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef worker fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class IN input
    class ORCH,SYNTH orch
    class W1,W2,W3 worker
    class OUT output
```

### Parallelization vs Orchestrator-Worker

| 비교 항목 | Parallelization | Orchestrator-Worker |
|-----------|----------------|---------------------|
| 태스크 수 | **코드에 미리 정의** | **LLM이 런타임에 결정** |
| 핵심 API | `add_edge(START, "node")` 다중 등록 | `Send` API로 동적 생성 |
| 유연성 | 낮음 (고정 분기) | 높음 (동적 분기) |
| 적합한 경우 | 고정된 분석 항목 | 입력에 따라 태스크 수가 달라질 때 |


In [ ]:
from langgraph.types import Send

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Orchestrator-Worker: Send API 기반 동적 분석
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → orchestrator → worker(병렬) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Orchestrator-Worker 실행
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 패턴 6: Evaluator-Optimizer (평가자-최적화자)

LLM이 생성한 결과를 **평가자 LLM**이 검토하고, 기준을 충족하지 못하면 **반복 개선**하는 패턴이에요.

```mermaid
flowchart TD
    IN["입력"] --> GEN["생성기 LLM<br>Generator"]
    GEN --> EVAL["평가자 LLM<br>Evaluator"]
    EVAL -->|"통과 PASS"| OUT["최종 출력"]
    EVAL -->|"개선 필요<br>FAIL + 피드백"| GEN

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef gen fill:#cce5ff,stroke:#007bff,color:#004085
    classDef eval fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class IN input
    class GEN gen
    class EVAL eval
    class OUT output
```


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Evaluator-Optimizer: 생성 + 자기 개선 루프
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → generate → evaluate → generate(루프) 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Evaluator-Optimizer 실행
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. Agent 패턴: 자율 에이전트

### 워크플로우와 뭐가 다른가요?

지금까지 본 6가지 워크플로우 패턴은 모두 **"다음에 뭘 할지"를 코드가 결정**했어요. 에이전트는 근본적으로 달라요. **LLM 스스로** 도구를 호출할지, 언제 멈출지를 결정해요. 마치 직원에게 "이 일을 알아서 처리해줘"라고 맡기는 것과 같아요. `should_continue` 함수가 에이전트 루프의 핵심이에요.

```mermaid
flowchart TD
    IN["입력"] --> LLM_NODE["LLM 노드<br/>call_llm"]
    LLM_NODE --> CHECK{"should_continue?<br/>tool_calls 확인"}
    CHECK -->|"도구 호출 있음"| TOOLS["도구 실행 노드<br/>tool_node"]
    CHECK -->|"도구 호출 없음<br/>(LLM이 충분하다고 판단)"| OUT["최종 응답"]
    TOOLS --> LLM_NODE

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef llm fill:#cce5ff,stroke:#007bff,color:#004085
    classDef tools fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef check fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef output fill:#f8d7da,stroke:#dc3545,color:#721c24

    class IN input
    class LLM_NODE llm
    class TOOLS tools
    class CHECK check
    class OUT output
```


In [ ]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langchain.messages import SystemMessage

    from datetime import datetime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Agent 패턴: tool loop + should_continue
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 실행: 도구 호출 과정 추적
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 패턴 선택 가이드

실제 문제를 만났을 때 어떤 패턴을 선택해야 할지 알아볼게요.


| 패턴 | 언제 사용? | 핵심 특징 | 주의사항 |
|------|-----------|-----------|----------|
| **Augmented LLM** | 도구나 검색이 필요한 단순 작업 | 도구 바인딩, 기본 블록 | 도구 수 5~10개 이내 |
| **Prompt Chaining** | 순차적 단계가 명확한 작업 | 품질 게이트, 단계별 전문화 | 오류 전파 방지 필요 |
| **Parallelization** | 독립적 서브태스크가 있는 작업 | Fan-out/Fan-in, operator.add | 결과 합산 reducer 필요 |
| **Routing** | 입력 유형에 따라 다른 처리가 필요 | 분류기 LLM, 전문 처리 경로 | 분류 실패 케이스 처리 |
| **Orchestrator-Worker** | 태스크 수를 미리 알 수 없는 경우 | Send API, 동적 워커 생성 | 워커 수 급증 시 비용 |
| **Evaluator-Optimizer** | 품질 기준이 있는 반복 개선 작업 | 피드백 루프, 자기 개선 | 최대 반복 횟수 설정 필수 |
| **Agent** | 비선형, 오픈엔드 복잡 작업 | should_continue, tool loop | 비용/지연 예측 어려움 |


In [ ]:
# ============================================================
# TODO: 패턴 선택 실습
#
# 아래 3가지 시나리오에 어떤 패턴이 적합한지 직접 구현해보세요.
# 힌트: 각 시나리오의 "제어 흐름이 고정인가? 유연한가?"를 판단해보세요.
#
# 시나리오 1: 사용자가 입력한 텍스트의 언어를 감지하고,
#             한국어면 영어로, 영어면 한국어로 번역 → Routing 패턴?
#
# 시나리오 2: 긴 보고서를 3개 섹션으로 나누어 각각 요약하고
#             최종 통합 요약 생성 → Parallelization 또는 Orchestrator-Worker?
#
# 시나리오 3: 사용자의 자유로운 질문에 답변하되,
#             필요하면 검색, 계산, 날씨 조회 등을 수행 → Agent 패턴?
#
# 예상 결과: 각 시나리오에 맞는 패턴을 선택하고 간단히 구현해보세요.
# ============================================================

# 여기에 코드를 작성해보세요!

# 예시: 시나리오 1 - 언어 감지 후 번역 (Routing 패턴 힌트)
# class TranslationState(TypedDict):
#     text: str
#     language: str  # 감지된 언어
#     translated: str
#
# def detect_language(state):
#     # 언어를 감지하는 LLM 호출
#     ...

# 직접 구현해보세요!


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **워크플로우 vs 에이전트**: 워크플로우는 제어 흐름이 코드에 고정, 에이전트는 LLM이 런타임에 결정
- **Augmented LLM**: 모든 패턴의 기반, `bind_tools`로 도구를 모델에 연결
- **Prompt Chaining**: 순차 단계 + 품질 게이트, 오류 전파 방지
- **Parallelization**: Fan-out/Fan-in, `Annotated[list, operator.add]` reducer로 결과 누적
- **Routing**: 분류기 LLM + `add_conditional_edges`로 전문 경로 분기
- **Orchestrator-Worker**: `Send` API로 동적 워커 생성, 태스크 수를 미리 알 수 없을 때
- **Evaluator-Optimizer**: 생성-평가-개선 피드백 루프, 최대 반복 횟수 필수
- **Agent 패턴**: `should_continue` + `tool loop`, `MessagesState` + `ToolNode` 활용


## 다음 노트북 예고

다음 `02-Design-Principles.ipynb`에서는 **LangGraph 설계 방법론 5단계**를 배워요. 워크플로우 매핑 → 타입 식별 → 상태 설계 → 노드 구현 → 엣지 연결 순서로 체계적으로 설계하는 방법, 4가지 에러 유형별 처리 전략, 그리고 흔한 안티패턴까지 다뤄요. 이 노트북에서 만난 6가지 패턴을 실무에 적용할 때 어디서 실수가 나오는지 미리 짚어볼게요.
